# Podcast RAG

## Overview

## Workflow

## Import necessary packages

Import all the necessary packages and libraries

In [1]:
import os
import torch
import logging
import requests
import feedparser
import torchaudio
import ipywidgets as widgets
from teapotai import TeapotAI
from urllib.parse import urlparse
from IPython.display import display
from huggingface_hub import notebook_login
from transformers import WhisperProcessor, WhisperForConditionalGeneration

logging.basicConfig(level=logging.INFO)

C:\Users\gta\Documents\seshu\02_Speech_n_Text\Final\my-own\AI-PC-Code-Samples\Audio-RAG\Podcast-RAG\.venv\Lib\site-packages\teapotai\teapotai.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
C:\Users\gta\Documents\seshu\02_Speech_n_Text\Final\my-own\AI-PC-Code-Samples\Audio-RAG\Podcast-RAG\.venv\Lib\site-packages\pydantic\_internal\_fields.py:198: UserWarning: Field name "schema" in "TeapotTool" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [2]:
def select_podcast_episode(PODCAST_URL):
    """
    Fetches and displays a dropdown widget to select an episode.

    Returns:
        dropdown: IPython widget for selecting podcast episode.
    """
    try:
        VALID_EXTENSIONS = ['.xml', '.rss']
        if any(PODCAST_URL.lower().endswith(ext) for ext in VALID_EXTENSIONS):
            logging.info(f" Found podcast URL.")    	
            feed = feedparser.parse(PODCAST_URL)
            episodes = []
            for entry in feed.entries:
                audio_url = None
                if not audio_url and hasattr(entry, 'enclosures') and entry.enclosures:
                    audio_url = entry.enclosures[0].href
                if audio_url:
                    episodes.append({'title': entry.title, 'url': audio_url})
            logging.info(f" Found {len(episodes)} episodes")
            episode_titles = [ep['title'] for ep in episodes]
            dropdown = widgets.Dropdown(
                options=episode_titles,
                value=episode_titles[0],
                description='Episode:'
            )
            display(dropdown)
            return dropdown, episodes
        else:
        	logging.info(f" Invalid podcast URL. Must end with one of: {VALID_EXTENSIONS}")
    except Exception as e:
        logging.exception(f" Error while selecting the podcast: {str(e)}")

In [3]:
PODCAST_URL = "https://feed.podbean.com/openatintel/feed.xml"
dropdown, episodes = select_podcast_episode(PODCAST_URL)

INFO:root: Found podcast URL.
INFO:root: Found 100 episodes


Dropdown(description='Episode:', options=('Building Innovation with Open Source AI', 'Democratizing Kubernetes…

In [4]:
def download_selected_audio(dropdown, episodes):
    """
    """
    try:
        OUTPUT_DIR = "downloads"
        selected_index = dropdown.index
        selected_url = episodes[selected_index]['url']
        logging.info(f" Selected episode: {episodes[selected_index]['title']}")
        logging.info(f" Audio URL: {selected_url}")
        logging.info(f" Downloading audio from: {selected_url}")
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        filename = os.path.basename(urlparse(selected_url).path)
        audio_path = os.path.join(OUTPUT_DIR, filename)
        response = requests.get(selected_url)
        with open(audio_path, 'wb') as f:
            f.write(response.content)
        logging.info(f" Audio saved to: {audio_path}")
        return audio_path
    except Exception as e:
        logging.exception(f" Error while downloading the podcast: {str(e)}")

In [5]:
audio_path = download_selected_audio(dropdown, episodes)

INFO:root: Selected episode: Building Innovation with Open Source AI
INFO:root: Audio URL: https://mcdn.podbean.com/mf/web/39af5nsnhjy8ckxr/OpenAtIntel_Ep111_MelissaMcKay-JFrog.mp3
INFO:root: Downloading audio from: https://mcdn.podbean.com/mf/web/39af5nsnhjy8ckxr/OpenAtIntel_Ep111_MelissaMcKay-JFrog.mp3
INFO:root: Audio saved to: downloads\OpenAtIntel_Ep111_MelissaMcKay-JFrog.mp3


In [6]:
def initialize_audio_models():
    """
    """
    try:
        model_id = "openai/whisper-base"
        device = "xpu" if torch.xpu.is_available() else "cpu"
        processor = WhisperProcessor.from_pretrained(pretrained_model_name_or_path=model_id)
        model = WhisperForConditionalGeneration.from_pretrained(pretrained_model_name_or_path=model_id)
        model = model.to(device)
        logging.info(" Model loaded!")
        return model, processor
    except Exception as e:
        logging.exception(f" Error while selecting the podcast: {str(e)}")

In [7]:
model, processor = initialize_audio_models()

INFO:root: Model loaded!


In [8]:
def process_podcast_audio(audio_path, model, processors):
    """
    """
    try:
        waveform, sample_rate = torchaudio.load(audio_path)
        logging.info(f" Original sample rate: {sample_rate} Hz")
        logging.info(f" Audio shape: {waveform.shape}")
        
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            logging.info(" Converted stereo to mono")
        
        if sample_rate != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
            waveform = resampler(waveform)
            sample_rate = 16000
            logging.info(" Resampled to 16kHz")
            logging.info(f" Audio shape: {waveform.shape}")
        
        audio = waveform.squeeze().numpy()
        logging.info(f" Audio duration: {len(audio)/sample_rate:.1f} seconds")
        chunk_length = 22 * 16000  # 22 seconds
        overlap_length = 1 * 16000  # 1 second overlap
        transcription_parts = []
        total_chunks = len(audio) // chunk_length + (1 if len(audio) % chunk_length > 0 else 0)
        logging.info(f" Processing {total_chunks} chunks of audio..")
        for i in range(0, len(audio), chunk_length - overlap_length):
            chunk = audio[i:i + chunk_length]
            if len(chunk) < 1600:  # Less than 0.1 seconds, skip that chunk
                continue
            logging.info(f" Processing chunk {len(transcription_parts) + 1}/{total_chunks}...")
            input_features = processor(chunk, sampling_rate=16000, return_tensors="pt").input_features
            input_features = input_features.to('xpu')
            with torch.no_grad():
                predicted_ids = model.generate(input_features)
            chunk_transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]        
            transcription_parts.append(chunk_transcription)
            logging.info(f" Chunk {len(transcription_parts)}: {chunk_transcription}..")
        transcription = " ".join(transcription_parts)
        logging.info(f"\n\n Total length: {len(transcription)} characters")
        logging.info(f" Audio path: {audio_path}")
        logging.info(f" Word count: {len(transcription.split())}")
        logging.info(transcription)
        return transcription_parts
    except Exception as e:
        logging.exception(f" Error while processing the input audio query : {str(e)}")

In [9]:
transcription_parts = process_podcast_audio(audio_path, model, processor)

INFO:root: Original sample rate: 44100 Hz
INFO:root: Audio shape: torch.Size([2, 64359798])
INFO:root: Converted stereo to mono
INFO:root: Resampled to 16kHz
INFO:root: Audio shape: torch.Size([1, 23350494])
INFO:root: Audio duration: 1459.4 seconds
INFO:root: Processing 67 chunks of audio..
INFO:root: Processing chunk 1/67...
INFO:root: Chunk 1:  My company was really interested in getting involved in this product called a Kia. So it's, you know, an attempt to standardize this. To put things together in such a way that everyone can do it the same, we can all contribute, we can all pay attention to the details. Welcome to the OpenItIntel podcast, where we're all about open source from software..
INFO:root: Processing chunk 2/67...
INFO:root: Chunk 2:  source from software to security to innovation and beyond. I'm your host, Catherine Druckmann, an open source evangelist at Intel, bringing you leading edge, free-ranging conversations from some of the best minds in the open source commun

## Generate Embeddings
These audio data chunks are embedded using the Teapot LLM to form a searchable knowledge base.

In [10]:
def generate_embeddings(transcription_parts):
    """
    Generate embeddings for the list of audio data chunks.
    
    Args:
        transcription_parts (list): Audio data chunks
    
    Returns:
        teapot_ai : Model with text embeddings

    Raises:
        Exception: Raises an exception if there is any error while generating embeddings for the audio data chunks.
    """
    try:
        if transcription_parts:
            logging.info(" Found transcriptions.")
            teapot_ai = TeapotAI(documents=transcription_parts)
            logging.info(" Generated embeddings.")
            return teapot_ai
        else:
            logging.info(" Did not find any transcriptions.")
    except Exception as e:
        logging.exception(f" Error while generating embeddings using TeapotAI : {str(e)}")    

In [11]:
teapot_ai = generate_embeddings(transcription_parts)

INFO:root: Found transcriptions.


 _____                      _         _    ___        __o__    _;;
|_   _|__  __ _ _ __   ___ | |_      / \  |_ _|   __ /-___-\__/ /
  | |/ _ \/ _` | '_ \ / _ \| __|    / _ \  | |   (  |       |__/
  | |  __/ (_| | |_) | (_) | |_    / ___ \ | |    \_|~~~~~~~|
  |_|\___|\__,_| .__/ \___/ \__/  /_/   \_\___|      \_____/
               |_|   
Loading Model


ERROR:root: Error while generating embeddings using TeapotAI : Cannot instantiate this tokenizer from a slow version. If it's based on sentencepiece, make sure you have sentencepiece installed.
Traceback (most recent call last):
  File "C:\Users\gta\AppData\Local\Temp\ipykernel_13868\2451067965.py", line 17, in generate_embeddings
    teapot_ai = TeapotAI(documents=transcription_parts)
  File "C:\Users\gta\Documents\seshu\02_Speech_n_Text\Final\my-own\AI-PC-Code-Samples\Audio-RAG\Podcast-RAG\.venv\Lib\site-packages\teapotai\teapotai.py", line 102, in __init__
    tokenizer = AutoTokenizer.from_pretrained(
        DEFAULT_MODEL,
        revision=DEFAULT_MODEL_REVISION
    )
  File "C:\Users\gta\Documents\seshu\02_Speech_n_Text\Final\my-own\AI-PC-Code-Samples\Audio-RAG\Podcast-RAG\.venv\Lib\site-packages\transformers\models\auto\tokenization_auto.py", line 1050, in from_pretrained
    return tokenizer_class.from_pretrained(pretrained_model_name_or_path, *inputs, **kwargs)
           ~~~~

In [12]:
%%time
# Get the answer using RAG

question = "Whos is the host here?"
answer = teapot_ai.chat([
    {
        "role":"system",
        "content": "You are an agent designed to answer the questions."
    },
    {
        "role":"user",
        "content": question
    }
])
print(answer)

CPU times: total: 0 ns
Wall time: 10.5 μs


AttributeError: 'NoneType' object has no attribute 'chat'

In [ ]:
%%time
# Get the answer using RAG

question = "what was the work of the host here"
answer = teapot_ai.chat([
    {
        "role":"system",
        "content": "You are an agent designed to answer the questions."
    },
    {
        "role":"user",
        "content": question
    }
])
print(answer)

In [ ]:
%%time
# Get the answer using RAG

question = "What is OPEA?"
answer = teapot_ai.chat([
    {
        "role":"system",
        "content": "You are an agent designed to answer the questions."
    },
    {
        "role":"user",
        "content": question
    }
])
print(answer)

In [ ]:
%%time
# Get the answer using RAG

question = "What is the podcast we are listening to?"
answer = teapot_ai.chat([
    {
        "role":"system",
        "content": "You are an agent designed to answer the questions."
    },
    {
        "role":"user",
        "content": question
    }
])
print(answer)

In [ ]:
%%time
# Get the answer using RAG

question = "What is the summary of the podcast?"
answer = teapot_ai.chat([
    {
        "role":"system",
        "content": "You are an agent designed to answer the questions."
    },
    {
        "role":"user",
        "content": question
    }
])
print(answer)